In [ ]:
# Paso 1: Instalación de dependencias
!pip install tensorflow

# Paso 2: Importación de bibliotecas necesarias
import tensorflow as tf
import numpy as np
import re
import unicodedata     #técnica de limpieza de datos

# Paso 3: Definición de funciones de preprocesamiento
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)
    w = re.sub(r"[^a-zA-Z?.!,¿]+", " ", w)
    w = w.strip()
    w = '<start> ' + w + ' <end>'
    return w



In [ ]:
# Paso 4: Creación de datos de ejemplo
conversaciones = [# Inteligencia Artificial y tu ejercicio actual
    ('¿por qué el archivo pesa 5 gigas?', 'porque contiene miles de imágenes en alta resolución.'),
    ('¿para qué sirve el encoder?', 'para comprimir la frase de entrada en un vector de contexto.'),
    ('¿puedo usar resnet para detectar frutas?', 'sí, solo debes congelar las capas y cambiar la salida.'),
    ('el modelo no carga en la ram.', 'intenta reducir el tamaño del batch o las unidades del encoder.'),

    # Control de movimiento y STM32
    ('¿cómo controlo el motor paso a paso?', 'puedes usar un driver a4988 y un stm32.'),
    ('el láser no está cortando bien.', 'revisa la alineación de los espejos y la potencia del tubo.'),
    ('¿qué tal va el proyecto de movimiento?', 'estoy ajustando los algoritmos de control para los servos.'),

    # Construcción y Techos
    ('¿qué ventajas tiene el metaldeck?', 'es muy rápido de instalar y sirve como formaleta.'),
    ('estoy terminando de montar el techo.', '¡excelente! el sistema metaldeck es muy seguro.'),

    # Agricultura (Cargamanto y Café)
    ('¿cuándo se cosecha el frijol cargamanto?', 'depende del clima, pero generalmente a los 4 meses.'),
    ('las gallinas se comen los huevos.', 'ponles nidos más oscuros o revisa su alimentación.'),
    ('¿vas a sembrar café en la sombra?', 'sí, es una buena técnica para mejorar la calidad del grano.')
]

# Paso 5: Preprocesamiento de los datos
input_texts = []
target_texts = []

for input_sentence, target_sentence in conversaciones:
    input_texts.append(preprocess_sentence(input_sentence))
    target_texts.append(preprocess_sentence(target_sentence))



In [ ]:
# Paso 6: Tokenización
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
tokenizer.fit_on_texts(input_texts + target_texts)

input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)

input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_sequences, padding='post')
target_tensor = tf.keras.preprocessing.sequence.pad_sequences(target_sequences, padding='post')

# Paso 7: Creación del conjunto de datos
BUFFER_SIZE = len(input_tensor)
BATCH_SIZE = 2
steps_per_epoch = len(input_tensor)//BATCH_SIZE
embedding_dim = 256
units = 1024
vocab_size = len(tokenizer.word_index)+1

dataset = tf.data.Dataset.from_tensor_slices((input_tensor, target_tensor)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)



In [ ]:
# Paso 8: Definición del modelo Encoder
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(self.enc_units,
                                         return_sequences=True,
                                         return_state=True,
                                         recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        return output, state_h, state_c

    def initialize_hidden_state(self):
        return [tf.zeros((self.batch_sz, self.enc_units)), tf.zeros((self.batch_sz, self.enc_units))]



In [ ]:
# Paso 9: Definición del mecanismo de atención Bahdanau
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        # query hidden state shape == (batch_size, hidden size)
        # values shape == (batch_size, max_len, hidden size)
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(
            self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights



In [ ]:
# Paso 10: Definición del modelo Decoder
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(self.dec_units,
                                         return_sequences=True,
                                         return_state=True,
                                         recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)

        # Mecanismo de atención
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        # x shape: (batch_size, 1)
        context_vector, attention_weights = self.attention(hidden[0], enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state_h, state_c = self.lstm(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, state_h, state_c, attention_weights



In [ ]:
# Paso 11: Inicialización de modelos y optimizador
encoder = Encoder(vocab_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(vocab_size, embedding_dim, units, BATCH_SIZE)

optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

# Paso 12: Definición de la función de pérdida
def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)



In [ ]:
# Paso 13: Definición del checkpoint
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = checkpoint_dir + '/ckpt'
checkpoint = tf.train.Checkpoint(optimizer=optimizer,
                                 encoder=encoder,
                                 decoder=decoder)



In [ ]:
# Paso 14: Entrenamiento del modelo con Early Stopping manual
EPOCHS = 50  # Puedes poner un número alto, el early stop lo detendrá
patience = 10  # Cuántas épocas esperar sin mejora antes de parar
wait = 0
best_loss = float('inf')

for epoch in range(EPOCHS):
    enc_hidden = encoder.initialize_hidden_state()
    total_loss = 0

    for (batch, (inp, targ)) in enumerate(dataset.take(steps_per_epoch)):
        # ... (aquí va todo tu código de entrenamiento actual) ...
        # ... (el GradientTape, el cálculo de gradientes y el optimizer) ...
        batch_loss = (loss / int(targ.shape[1]))
        total_loss += batch_loss

    # CALCULAMOS LA PÉRDIDA PROMEDIO DE LA ÉPOCA
    current_loss = total_loss / steps_per_epoch
    print(f'Epoch {epoch+1}/{EPOCHS}, Loss: {current_loss:.4f}')

    # --- LÓGICA DE EARLY STOPPING ---
    if current_loss < best_loss:
        best_loss = current_loss
        wait = 0
        # Opcional: Guardar el mejor modelo aquí
        # checkpoint.save(file_prefix=checkpoint_prefix)
    else:
        wait += 1
        print(f'Sin mejora. Esperando... {wait}/{patience}')
        if wait >= patience:
            print(f'Early Stopping activado. El entrenamiento se detiene en la época {epoch+1}')
            break




In [ ]:
# Paso 15: Función para evaluar nuevas frases
def evaluate(sentence):
    sentence = preprocess_sentence(sentence)
    inputs = [tokenizer.word_index.get(w, tokenizer.word_index['<start>']) for w in sentence.split(' ')]
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs], maxlen=input_tensor.shape[1], padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''

    enc_hidden = [tf.zeros((1, units)), tf.zeros((1, units))]
    enc_out, enc_hidden_h, enc_hidden_c = encoder(inputs, enc_hidden)

    dec_hidden = [enc_hidden_h, enc_hidden_c]
    dec_input = tf.expand_dims([tokenizer.word_index['<start>']], 0)

    for t in range(target_tensor.shape[1]):
        predictions, dec_hidden_h, dec_hidden_c, attention_weights = decoder(dec_input, dec_hidden, enc_out)
        dec_hidden = [dec_hidden_h, dec_hidden_c]

        predicted_id = tf.argmax(predictions[0]).numpy()

        if tokenizer.index_word[predicted_id] == '<end>':
            return result.strip()

        result += tokenizer.index_word[predicted_id] + ' '

        dec_input = tf.expand_dims([predicted_id], 0)

    return result.strip()

# Paso 16: Prueba del modelo
test_sentence = '¿has visitado Medellín?'
response = evaluate(test_sentence)
print(f'Entrada: {test_sentence}')
print(f'Respuesta: {response}')
